# Road Follower - Train Model

In this notebook we will train a neural network to take an input image, and output a set of x, y values corresponding to a target.

We will be using PyTorch deep learning framework to train ResNet18 neural network architecture model for road follower application.

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
import glob
import PIL.Image
import os
import numpy as np

Place the labelled `dataset_xy` folder next to this notebook, then run the next cell. The original unzip cell assumed a fixed ZIP filename.


In [ ]:
from pathlib import Path
images = sorted(Path('dataset_xy').glob('*.jpg'))
print('Training images:', len(images))
assert len(images) >= 20, 'Put labelled images in dataset_xy first (at least 20 to run).'


The next cell should report at least 20 labelled JPG images.


### Create Dataset Instance

Here we create a custom ``torch.utils.data.Dataset`` implementation, which implements the ``__len__`` and ``__getitem__`` functions.  This class
is responsible for loading images and parsing the x, y values from the image filenames.  Because we implement the ``torch.utils.data.Dataset`` class,
we can use all of the torch data utilities :)

We hard coded some transformations (like color jitter) into our dataset.  We made random horizontal flips optional (in case you want to follow a non-symmetric path, like a road
where we need to 'stay right').  If it doesn't matter whether your robot follows some convention, you could enable flips to augment the dataset.

In [ ]:
def get_x(path, width):
    return (float(int(path.split('_')[1])) - width / 2) / (width / 2)

def get_y(path, height):
    return (float(int(path.split('_')[2])) - height / 2) / (height / 2)

class XYDataset(torch.utils.data.Dataset):
    def __init__(self, directory, random_hflips=False, augment=False):
        self.image_paths = sorted(glob.glob(os.path.join(directory, '*.jpg')))
        self.random_hflips = random_hflips
        self.augment = augment
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = PIL.Image.open(image_path).convert('RGB')
        width, height = image.size
        name = os.path.basename(image_path)
        x = get_x(name, width)
        y = get_y(name, height)
        if self.random_hflips and np.random.rand() > 0.5:
            image = transforms.functional.hflip(image)
            x = -x
        if self.augment:
            image = self.color_jitter(image)
        image = transforms.functional.resize(image, (224, 224))
        image = transforms.functional.to_tensor(image)
        # Preserve the original notebook's BGR channel order and normalization.
        image = torch.from_numpy(image.numpy()[::-1].copy())
        image = transforms.functional.normalize(image,
            [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return image, torch.tensor([x, y]).float()

dataset = XYDataset('dataset_xy')
print('Examples:', len(dataset))


### Split dataset into train and test sets
Once we read dataset, we will split data set in train and test sets. In this example we split train and test a 90%-10%. The test set will be used to verify the accuracy of the model we train.

In [ ]:
test_percent = 0.1
indices = np.random.RandomState(42).permutation(len(dataset))
num_test = max(1, int(test_percent * len(dataset)))
train_dataset = torch.utils.data.Subset(
    XYDataset('dataset_xy', random_hflips=False, augment=True), indices[num_test:])
test_dataset = torch.utils.data.Subset(
    XYDataset('dataset_xy', random_hflips=False, augment=False), indices[:num_test])
print('Train:', len(train_dataset), 'Validation:', len(test_dataset))


### Create data loaders to load data in batches

We use ``DataLoader`` class to load data in batches, shuffle data and allow using multi-subprocesses. In this example we use batch size of 64. Batch size will be based on memory available with your GPU and it can impact accuracy of the model.

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

### Define Neural Network Model 

We use ResNet-18 model available on PyTorch TorchVision. 

In a process called transfer learning, we can repurpose a pre-trained model (trained on millions of images) for a new task that has possibly much less data available.


More details on ResNet-18 : https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py

More Details on Transfer Learning: https://www.youtube.com/watch?v=yofjFQddwHE 

In [ ]:
try:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
except AttributeError:
    model = models.resnet18(pretrained=True)


ResNet model has fully connect (fc) final layer with 512 as ``in_features`` and we will be training for regression thus ``out_features`` as 1

Finally, we transfer our model for execution on the GPU

In [ ]:
model.fc = torch.nn.Linear(512, 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model = model.to(device)


### Train Regression:

We train for 50 epochs and save best model if the loss is reduced. 

In [ ]:
NUM_EPOCHS = 70
BEST_MODEL_PATH = 'best_steering_model_xy.pth'
best_loss = float('inf')
optimizer = optim.Adam(model.parameters())

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.mse_loss(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            test_loss += F.mse_loss(model(images), labels).item()
    test_loss /= len(test_loader)
    print('Epoch {:02d}/{:02d}  train={:.5f}  validation={:.5f}'.format(
        epoch + 1, NUM_EPOCHS, train_loss, test_loss))
    if test_loss < best_loss:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_loss = test_loss
print('Saved:', BEST_MODEL_PATH, 'best validation loss:', best_loss)


Once the model is trained, it will generate ``best_steering_model_xy.pth`` file which you can use for inferencing in the live demo notebook.

If you trained on a different machine other than JetBot, you'll need to upload this to the JetBot to the ``road_following`` example folder.